In [1]:
## Mirrors the cellPLM-/scGPT-/Geneformer-embedding-wrapper notebooks: load a
## dataset, pad+reorder genes to scFoundation's fixed 19,264-gene vocabulary,
## run zero-shot per-cell inference with the 100M-encoder checkpoint, and
## write the per-cell embedding to an h5ad keyed by obs_names so
## benchmark.ipynb can align it independently of cell order.
##
## NOTE: scFoundation runs one cell per forward pass (no single-cell batching).
## On 48k cells this can take ~30–90 minutes on a Blackwell GPU. Set DEMO=True
## below to quickly sanity-check on the first 50 cells before committing.

In [2]:
import sys, os
SF_MODEL_DIR = os.path.abspath('../../models/scFoundation/model')
sys.path.insert(0, SF_MODEL_DIR)

import warnings
warnings.filterwarnings('ignore')

import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import anndata as ad
from scipy.sparse import issparse
from tqdm import tqdm

# These come from scFoundation/model/load.py — only available once SF_MODEL_DIR
# is on sys.path and the conda env from setup_scfoundation.sh is active.
from load import load_model_frommmf, gatherData

In [4]:
DATA_PATH    = Path('../../data/cellPLM/data/gse155468.h5ad')
SF_REPO_DIR  = Path('../../models/scFoundation/model').resolve()
CKPT_PATH    = SF_REPO_DIR / 'models' / 'models.ckpt'   # placed there by setup_scfoundation.sh
GENE_INDEX   = SF_REPO_DIR / 'OS_scRNA_gene_index.19264.tsv'
DEVICE       = 'cuda:0'

# Knobs that match scFoundation's get_embedding.py defaults for cell embeddings:
#   VERSION='ce'     : pretrained 'cell embedding' head (vs 'rde' for read-depth enhancement)
#   POOL_TYPE='all'  : concat [last-tok, 2nd-last-tok, max-pool, mean-pool] -> 4*hidden_dim
#   TGTHIGHRES='a5'  : T = log10(total_count) + 5  (target high-resolution token; readme default is 't4')
#   PRE_NORMALIZED='F': raw counts in, scFoundation does its own normalize+log1p per cell
VERSION        = 'ce'
POOL_TYPE      = 'all'
TGTHIGHRES     = 'a5'
PRE_NORMALIZED = 'F'

DEMO = False     # True -> only embed the first 50 cells (smoke test)
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

assert CKPT_PATH.exists(), (
    f'Missing checkpoint: {CKPT_PATH}\n'
    f'Run env/setup_scfoundation.sh first and follow its manual-download instructions.'
)

In [5]:
# Load the source dataset and reshape to scFoundation's expected (n_cells, 19264)
# layout: rows are cells, columns are gene SYMBOLS in the canonical order from
# OS_scRNA_gene_index.19264.tsv. Genes that gse155468 doesn't have get padded
# with zeros; gse155468 genes outside the 19264 set are dropped.
data = ad.read_h5ad(DATA_PATH)
data.obs_names_make_unique()

# Build a (cells x gene_symbols) DataFrame from raw counts.
X_raw = data.X.toarray() if issparse(data.X) else data.X
X_df  = pd.DataFrame(X_raw, index=data.obs_names, columns=data.var_names).astype(np.float32)

# scFoundation's canonical gene list (19,264 protein-coding gene SYMBOLS).
gene_list = pd.read_csv(GENE_INDEX, header=0, delimiter='\t')['gene_name'].tolist()

# Pad missing genes with zeros and reorder to the canonical 19264 sequence.
# This is exactly what get_embedding.py's main_gene_selection() does.
missing  = [g for g in gene_list if g not in X_df.columns]
padding  = pd.DataFrame(np.zeros((X_df.shape[0], len(missing)), dtype=np.float32),
                        index=X_df.index, columns=missing)
X_full   = pd.concat([X_df, padding], axis=1)[gene_list]
n_overlap = X_df.shape[1] - len(set(X_df.columns) - set(gene_list))
print(f'gse155468 genes: {X_df.shape[1]}, overlap with scFoundation vocab: {n_overlap}, padded: {len(missing)}')
print(f'reshaped count matrix: {X_full.shape}')

if DEMO:
    X_full = X_full.iloc[:50, :]
    print(f'DEMO=True -> embedding only first {X_full.shape[0]} cells')

gse155468 genes: 12382, overlap with scFoundation vocab: 10750, padded: 8514
reshaped count matrix: (48082, 19264)


In [6]:
# Load the pretrained model. The 'cell' key picks the cell-embedding head
# (vs 'rde' for read-depth enhancement, or 'gene' for per-gene embeddings).
key = 'cell' if VERSION == 'ce' else ('rde' if VERSION == 'rde' else None)
assert key is not None, f'Unsupported VERSION={VERSION!r} for cell embedding'

pretrainmodel, pretrainconfig = load_model_frommmf(str(CKPT_PATH), key)
pretrainmodel = pretrainmodel.to(DEVICE).eval()
PAD_TOKEN_ID  = pretrainconfig['pad_token_id']
HIDDEN        = pretrainmodel.token_emb.weight.shape[1] if hasattr(pretrainmodel.token_emb, 'weight') else None
print(f'loaded scFoundation: pad_token_id={PAD_TOKEN_ID}, hidden_dim≈{HIDDEN}')

{'mask_gene_name': False, 'gene_num': 19266, 'seq_len': 19266, 'encoder': {'hidden_dim': 768, 'depth': 12, 'heads': 12, 'dim_head': 64, 'seq_len': 19266, 'module_type': 'transformer', 'norm_first': False}, 'decoder': {'hidden_dim': 512, 'depth': 6, 'heads': 8, 'dim_head': 64, 'module_type': 'performer', 'seq_len': 19266, 'norm_first': False}, 'n_class': 104, 'pad_token_id': 103, 'mask_token_id': 102, 'bin_num': 100, 'bin_alpha': 1.0, 'rawcount': True, 'model': 'mae_autobin', 'test_valid_train_idx_dict': '/nfs_beijing/minsheng/data/os10000w-new/global_shuffle/meta.csv.train_set_idx_dict.pt', 'valid_data_path': '/nfs_beijing/minsheng/data/valid_count_10w.npz', 'num_tokens': 13, 'train_data_path': None, 'isPanA': False, 'isPlanA1': False, 'max_files_to_load': 5, 'bin_type': 'auto_bin', 'value_mask_prob': 0.3, 'zero_mask_prob': 0.03, 'replace_prob': 0.8, 'random_token_prob': 0.1, 'mask_ignore_token_ids': [0], 'decoder_add_zero': True, 'mae_encoder_max_seq_len': 15000, 'isPlanA': False, 'ma

In [7]:
# Per-cell inference. Mirrors the singlecell branch of get_embedding.py exactly:
#   1. log1p-normalize counts to 1e4 totals (when PRE_NORMALIZED='F')
#   2. append [T, S] = [target_high_res_token, log10(total_count)] to make a length-19266 input
#   3. token_emb + pos_emb -> encoder
#   4. pool the per-token hidden states into one cell vector (per POOL_TYPE)
# Output: (n_cells, 4*hidden_dim) when POOL_TYPE='all', else (n_cells, hidden_dim).

embs = []
n_cells = X_full.shape[0]
data_gene_ids_template = torch.arange(19266, device=DEVICE)

with torch.no_grad():
    for i in tqdm(range(n_cells), desc='scFoundation infer'):
        row = X_full.iloc[i, :].to_numpy()

        # 1. normalize
        if PRE_NORMALIZED == 'F':
            total = row.sum()
            tmp   = np.log1p(row / max(total, 1.0) * 1e4).tolist()
        elif PRE_NORMALIZED == 'T':
            total = row.sum()
            tmp   = row.tolist()
        elif PRE_NORMALIZED == 'A':
            total = row[-1]
            tmp   = row[:-1].tolist()
        else:
            raise ValueError(f'PRE_NORMALIZED must be F/T/A, got {PRE_NORMALIZED!r}')

        # 2. append [T, S]
        log10_total = float(np.log10(max(total, 1.0)))
        if   TGTHIGHRES[0] == 'a':
            t_token = log10_total + float(TGTHIGHRES[1:])
        elif TGTHIGHRES[0] == 'f':
            t_token = float(np.log10(max(total, 1.0) * float(TGTHIGHRES[1:])))
        elif TGTHIGHRES[0] == 't':
            t_token = float(TGTHIGHRES[1:])
        else:
            raise ValueError(f'TGTHIGHRES must start with a/f/t, got {TGTHIGHRES!r}')

        pretrain_gene_x = torch.tensor(tmp + [t_token, log10_total],
                                       dtype=torch.float32, device=DEVICE).unsqueeze(0)
        data_gene_ids = data_gene_ids_template.unsqueeze(0)

        # 3. gather non-zero entries (sparse-attention path) and run encoder
        value_labels = pretrain_gene_x > 0
        x, x_padding = gatherData(pretrain_gene_x, value_labels, PAD_TOKEN_ID)
        position_gene_ids, _ = gatherData(data_gene_ids, value_labels, PAD_TOKEN_ID)

        x = pretrainmodel.token_emb(torch.unsqueeze(x, 2).float(), output_weight=0)
        x = x + pretrainmodel.pos_emb(position_gene_ids)
        geneemb = pretrainmodel.encoder(x, x_padding)

        # 4. pool — last 2 tokens are the [T, S] high-res tokens
        e_last      = geneemb[:, -1, :]                            # T-token rep
        e_second    = geneemb[:, -2, :]                            # S-token rep
        e_max, _    = torch.max (geneemb[:, :-2, :], dim=1)        # max over genes
        e_mean      = torch.mean(geneemb[:, :-2, :], dim=1)        # mean over genes
        if POOL_TYPE == 'all':
            cell_emb = torch.cat([e_last, e_second, e_max, e_mean], dim=1)
        elif POOL_TYPE == 'max':
            cell_emb, _ = torch.max(geneemb, dim=1)
        else:
            raise ValueError(f'POOL_TYPE must be all/max, got {POOL_TYPE!r}')
        embs.append(cell_emb.detach().cpu().numpy())

embedding = np.squeeze(np.array(embs)).astype(np.float32)
if embedding.ndim == 1:
    embedding = embedding[None, :]
print('embedding shape:', embedding.shape, 'dtype:', embedding.dtype)
assert embedding.shape[0] == n_cells, (embedding.shape, n_cells)

scFoundation infer: 100%|██████████| 48082/48082 [32:45<00:00, 24.47it/s] 


embedding shape: (48082, 3072) dtype: float32


In [8]:
# Match the cellPLM-/scGPT-/Geneformer-embedding-wrapper output layout so
# benchmark.ipynb can align by obs_names: X = (n_cells, d) float32,
# obs_names = original cell IDs, var_names = ['dim_0', 'dim_1', ...].
embedding_adata = ad.AnnData(X=embedding)
embedding_adata.obs_names = X_full.index   # preserved through the per-cell loop
embedding_adata.var_names = [f'dim_{i}' for i in range(embedding.shape[1])]
embedding_adata.write_h5ad('gse155468_embedding.h5ad')
print('wrote gse155468_embedding.h5ad', embedding_adata.shape)

wrote gse155468_embedding.h5ad (48082, 3072)
